# 01 — Exploration des données brutes du Fournil Vert

**Objectif** : découvrir les 4 sources de la version minimale et repérer ce qui empêche de les utiliser telles quelles.

Ce notebook **explore** : on ouvre, on regarde, on note. La correction sera faite dans un **script** (`scripts/02_clean_ventes.py`), approfondi en séquence 4.

In [1]:
import json
import pandas as pd

RAW = "../data/raw/"

## 1. Caisse de la boutique 001 (Puteaux) — CSV

In [2]:
caisse_a = pd.read_csv(RAW + "caisse_boutique_A.csv", sep=";", encoding="utf-8-sig")
caisse_a.head()

,ticket_id,date_vente,heure,code_boutique,produit,famille,quantite,montant_ttc
0,A260309-0001,2026-03-09,07:02,1,Baguette tradition,Pains,7,"9,10"
1,A260309-0002,2026-03-09,07:03,1,Sandwich jambon-beurre,Snacking,1,"4,90"
2,A260309-0003,2026-03-09,07:05,1,Croissant,Viennoiseries,1,"1,20"
3,A260309-0003,2026-03-09,07:05,1,Pain au chocolat,Viennoiseries,1,"1,40"
4,A260309-0004,2026-03-09,07:05,1,Baguette tradition,Pains,3,"3,90"


In [3]:
caisse_a.dtypes

ticket_id        object
date_vente       object
heure            object
code_boutique     int64
produit          object
famille          object
quantite          int64
montant_ttc      object
dtype: object

**Constat 1 — type incorrect** : `montant_ttc` est de type `object` (du texte), à cause de la virgule décimale. Impossible de faire une somme fiable en l'état.

Autre piège : `code_boutique` est lu comme un **nombre** par pandas. Le code `001` devient `1` : les zéros sont perdus.

In [4]:
# Nombre de lignes par jour : le samedi 14 a deux fois plus de lignes que les autres jours
caisse_a["date_vente"].value_counts().sort_index()

date_vente
2026-03-09    207
2026-03-10    204
2026-03-11    192
2026-03-12    183
2026-03-13    200
2026-03-14    564
2026-03-15    290
Name: count, dtype: int64

In [5]:
caisse_a.duplicated().sum()

np.int64(282)

**Constat 2 — doublons** : toutes les lignes du samedi 14 mars sont présentes deux fois (export de caisse relancé). Sans correction, le CA du samedi est doublé.

## 2. Caisse de la boutique 004 (Suresnes) — CSV d'un autre logiciel

In [6]:
caisse_b = pd.read_csv(RAW + "caisse_boutique_B.csv", sep=";", encoding="utf-8-sig")
caisse_b.head()

,Date,Heure,N° ticket,Magasin,Article,Rayon,Qté,Total TTC
0,09/03/2026,07:09,50232,4,Sandwich jambon-beurre,SNACKING,1,490
1,09/03/2026,07:11,50233,4,Sandwich jambon-beurre,SNACKING,1,490
2,09/03/2026,07:15,50234,4,Baguette tradition,PAIN,1,130
3,09/03/2026,07:19,50235,4,Croissant,VIENNOISERIE,1,120
4,09/03/2026,07:23,50236,4,Éclair café,PATISSERIE,1,280


In [7]:
print("Colonnes caisse A :", list(caisse_a.columns))
print("Colonnes caisse B :", list(caisse_b.columns))

Colonnes caisse A : ['ticket_id', 'date_vente', 'heure', 'code_boutique', 'produit', 'famille', 'quantite', 'montant_ttc']
Colonnes caisse B : ['Date', 'Heure', 'N° ticket', 'Magasin', 'Article', 'Rayon', 'Qté', 'Total TTC']


In [8]:
caisse_b["Total TTC"].describe()

count    1186.000000
mean      327.436762
std       285.746646
min       120.000000
25%       130.000000
50%       260.000000
75%       420.000000
max      1780.000000
Name: Total TTC, dtype: float64

**Constat 3 — noms de colonnes différents** : `date_vente` / `Date`, `montant_ttc` / `Total TTC`, `famille` / `Rayon`…

**Constat 4 — unités différentes** : un sandwich à 4,90 € vaut `490` dans la caisse B. Les montants sont en **centimes**.

**Constat 5 — dates incompatibles** : `2026-03-09` dans la caisse A, `09/03/2026` dans la caisse B.

## 3. Production et invendus — Excel saisi par les gérants

In [9]:
prod = pd.read_excel(RAW + "production_invendus.xlsx")
prod.head(8)

,Date,Code boutique,Famille,Qté produite,Invendus,CA
0,2026-03-09 00:00:00,1,Pains,110,11.0,141.9
1,2026-03-09 00:00:00,1,Viennoiseries,112,15.0,128.0
2,2026-03-09 00:00:00,1,Snacking,71,7.0,362.4
3,2026-03-09 00:00:00,1,Pâtisseries,19,2.0,33.6
4,2026-03-09 00:00:00,4,Pains,81,13.0,117.0
5,2026-03-09 00:00:00,4,Viennoiseries,63,11.0,67.8
6,2026-03-09 00:00:00,4,Snacking,36,5.0,156.2
7,2026-03-09 00:00:00,4,Pâtisseries,20,3.0,35.6


In [10]:
# Les codes boutique : pandas les lit comme des nombres, les zéros disparaissent
prod["Code boutique"].value_counts()

Code boutique
1    28
4    28
Name: count, dtype: int64

In [11]:
# Dates : une partie a été saisie comme un nombre (46093 = 12/03/2026 au format Excel)
prod["Date"].map(type).value_counts()

Date
<class 'datetime.datetime'>    52
<class 'int'>                   4
Name: count, dtype: int64

In [12]:
prod[prod["Invendus"].isna()]

,Date,Code boutique,Famille,Qté produite,Invendus,CA
52,2026-03-15 00:00:00,4,Pains,99,NaN,147.4
53,2026-03-15 00:00:00,4,Viennoiseries,81,NaN,94.0
54,2026-03-15 00:00:00,4,Snacking,104,NaN,571.4
55,2026-03-15 00:00:00,4,Pâtisseries,61,NaN,120.8


**Constat 6 — valeurs manquantes** : les invendus du dimanche 15 mars ne sont pas saisis pour la boutique 4.

**Types incorrects, encore** : dans l'Excel, le gérant de Suresnes a saisi `4` au lieu de `004` (ouvre le fichier pour le voir), et pandas transforme de son côté `001` en `1`. Solution : lire les codes en texte (`dtype={"Code boutique": str}`) puis les compléter à 3 caractères. Une date est aussi restée un nombre.

## 4. Météo — JSON renvoyé par l'API

In [13]:
with open(RAW + "meteo_extrait.json", encoding="utf-8") as f:
    meteo = json.load(f)

# Un JSON est un arbre : une liste de villes, chacune avec ses mesures « daily »
print(meteo[0]["ville"], list(meteo[0]["daily"].keys()))

Puteaux ['time', 'temperature_2m_max', 'precipitation_sum']


In [14]:
# Mise à plat en tableau pour une ville
pd.DataFrame(meteo[0]["daily"])

,time,temperature_2m_max,precipitation_sum
0,2026-03-09,11.2,4.6
1,2026-03-10,12.8,0.0
2,2026-03-11,14.5,0.0
3,2026-03-12,9.7,8.2
4,2026-03-13,13.1,1.4
5,2026-03-14,17.9,0.0
6,2026-03-15,16.4,0.0


## Bilan

| Défaut | Où | Correction prévue |
| --- | --- | --- |
| Doublons | Caisse A, samedi 14 | Supprimer les lignes identiques |
| Valeurs manquantes | Excel, dimanche 15, boutique 4 | Les signaler, ne pas les remplacer par 0 |
| Dates incompatibles | Caisses A et B, Excel | Un seul format de date |
| Noms de colonnes différents | Caisses A et B | Renommer avec un nom commun |
| Unités différentes | Caisse B en centimes | Diviser par 100 |
| Types incorrects | Montants en texte, code `4` | Convertir en nombre / en texte sur 3 caractères |

→ Une fois corrigées, ces données donnent le **dataset final** : 1 ligne = 1 boutique × 1 jour.